### Setup en imports
imports, logging setup, en de SDM aanmaken vanuit het RIM script.

In [36]:
import sqlite3
import shutil
import pandas as pd
from loguru import logger

logger.remove()
logger.add("pipeline.log", level="DEBUG")

SDM_DB = "GO_SDM.db"
RIM_SCRIPT = "sdm_RIM.txt"

def create_sdm_db():

    with open(RIM_SCRIPT, "r") as f:
        sql = f.read()

    conn = sqlite3.connect(SDM_DB)
    try:
        conn.executescript(sql)
        conn.commit()
        logger.info("SDM database aangemaakt op basis van RIM script")
    except Exception as e:
        logger.error(f"fout bij aanmaken SDM database: {e}")
        raise
    finally:
        conn.close()

create_sdm_db()

### Reset SDM
Maakt alle tabellen in de SDM leeg zodat we opnieuw kunnen laden zonder conflict met foreign keys
Inlaadstrategie: Full Refresh + Schema Driven Bulk Loading

In [44]:
def reset_sdm():
    conn = sqlite3.connect(SDM_DB)
    cur = conn.cursor()

    try:
        cur.execute("PRAGMA foreign_keys = OFF")
        cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tabellen = [row[0] for row in cur.fetchall()]

        for tabel in tabellen:
            cur.execute(f"DELETE FROM {tabel}")
            logger.info(f"Tabel '{tabel}' leeggemaakt.")

        cur.execute("PRAGMA foreign_keys = ON")
        conn.commit()
        logger.info("SDM reset succesvol afgerond.")
    except Exception as e:
        logger.error(f"fout bij resetten SDM: {e}")
        raise
    finally:
        conn.close()

reset_sdm()

### GO_SALES
Full Refresh vanuit GO_SALES-data.sqlite naar de SDM. Alle tabellen worden leeggemaakt en opnieuw ingevuld.

In [ ]:
GO_SALES_DB = "../data/GO_SALES-data.sqlite"

koppeling_go_sales = {
    "order_method"  : "order_method",
    "return_reason" : "return_reason",
    "product_line"  : "product_line",
    "country"       : "country",
    "product_type"  : "product_type",
    "sales_branch"  : "sales_branch",
    "product"       : "product",
    "retailer_site" : "retailer_site",
    "sales_staff"   : "sales_staff",
    "order_header"  : "Order_Header",
    "order_details" : "order_details",
    "returned_item" : "returned_item"
}

conn_sdm = sqlite3.connect(SDM_DB)
cur_sdm = conn_sdm.cursor()
logger.info("Connectie met SDM database geopend.")

conn_source = sqlite3.connect(GO_SALES_DB)
cur_source = conn_source.cursor()
logger.info(f"SDM wordt ingevuld uit database: {GO_SALES_DB}")

try:
    for tabel_source, tabel_sdm in koppeling_go_sales.items():

        cur_source.execute(f"SELECT * FROM {tabel_source}")
        data = cur_source.fetchall()
        logger.info(f"SDM tabel '{tabel_sdm}' wordt ingevuld vanuit operationele tabel '{tabel_source}'.")

        placeholders = ", ".join(["?"] * len(data[0]))
        cur_sdm.executemany(f"INSERT INTO {tabel_sdm} VALUES ({placeholders})", data)
        logger.info(f"SDM tabel '{tabel_sdm}' is nu ingevuld met {len(data)} rijen")

    conn_sdm.commit()

except Exception as e:
    logger.error(f"Fout bij laden GO_SALES: {e}")
    raise
finally:
    conn_source.close()
    logger.info(f"Connectie met operationele database '{GO_SALES_DB}' gesloten.")
    conn_sdm.close()
    logger.info("Connectie met SDM database gesloten.")

<h1>CRM-data</h1>

In [14]:
CRM_DATA_DB = "../data/CRM-data.sqlite"

koppeling_crm_data = {
    "sales_territory" : "Sales_Territory",
    "age_group" : "Age_Group",
    "crm_country" : "Crm_Country",
    "sales_demographic" : "Sales_Demographic" ,
    "customer_segment" : "Customer_Segment",
    "customer_type" : "Customer_Type",
    "customer_contact" : "Customer_Contact",
    "customer_store" : "Customer_Store",
    "customer" : "Customer",
    "customer_headquarters" : "Customer_Headquarters"
}

conn_sdm = sqlite3.connect(SDM_DB)
cur_sdm = conn_sdm.cursor()
logger.info("Connectie met SDM database geopend.")

conn_source = sqlite3.connect(CRM_DATA_DB)
cur_source = conn_source.cursor()
logger.info(f"SDM wordt ingevuld uit database: {CRM_DATA_DB}")

try:
    for tabel_source, tabel_sdm in koppeling_crm_data.items():
        cur_source.execute(f"SELECT * FROM {tabel_source}")
        data = cur_source.fetchall()
        logger.info(f"SDM tabel '{tabel_sdm}' wordt ingevuld vanuit operationele tabel '{tabel_source}'.")
        
        placeholders = ", ".join(["?"] * len(data[0]))
        cur_sdm.executemany(f"INSERT INTO {tabel_sdm} VALUES ({placeholders})", data)
        logger.info(f"SDM tabel '{tabel_sdm}' is nu ingevuld met {len(data)} rijen")
    
    conn_sdm.commit()

except Exception as e:
    logger.error(f"Fout bij laden GO_SALES: {e}")
    raise
finally:
    conn_source.close()
    logger.info(f"Connectie met operationele database '{CRM_DATA_DB}' gesloten.")
    conn_sdm.close()
    logger.info("Connectie met SDM database gesloten.")

<h1>Inventory Levels</h1>

In [ ]:
INVENTORY_LEVELS_CSV = "../data/INVENTORY_LEVELS-data.csv"
tabel_naam = "Inventory_Levels"

df_inv_levels = pd.read_csv(INVENTORY_LEVELS_CSV, dtype ={
    "INVENTORY_YEAR": "int64",
    "INVENTORY_MONTH": "int64",
    "PRODUCT_NUMBER": "int64",
    "INVENTORY_COUNT": "int64"
})

conn_sdm = sqlite3.connect(SDM_DB)
cursor = conn_sdm.cursor()
logger.info("Connectie met SDM database geopend.")

df_inv_levels.to_sql(
    name = tabel_naam,
    con = conn_sdm,
    if_exists = "append",
    index = False
)

cursor.execute("SELECT COUNT(*) FROM Inventory_Levels")
rijen = cursor.fetchone()[0]
logger.info(f"SDM tabel '{tabel_naam}' is nu ingevuld met {rijen} rijen")

conn_sdm.close()
logger.info("Connectie met SDM database gesloten.")

<h1>Product Forecast</h1>

<h1>Sales Target</h1>